# 📊 YUVAAN — EDA + CBES v2 Scoring
### Indian Loan Dataset | Data Analysis & Credit Scoring
---

## Step 1: Imports & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset — update path if needed
df = pd.read_csv('synthetic_indian_loan_dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nDefault distribution:')
print(df['default_risk'].value_counts())
print(f'\nDefault rate: {df["default_risk"].mean()*100:.1f}%')
df.head(3)

## Step 2: Data Cleaning

In [ ]:
# No missing values in this dataset
print('Missing values:')
print(df.isnull().sum().sum(), 'total missing values')

# Add readable columns
df['AMT_INCOME_K'] = df['annual_income'] / 1000
df['AMT_CREDIT_K'] = df['loan_amount'] / 1000

print('\nKey stats:')
print(df[['annual_income','loan_amount','cibil_score','missed_payments','emi_income_ratio']].describe().round(2))

## Step 3: EDA — 4 Graphs

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Indian Loan Dataset — EDA', fontsize=15, fontweight='bold')

# --- Graph 1: Default vs Non-Default Count ---
ax1 = axes[0, 0]
counts = df['default_risk'].value_counts().sort_index()
bars = ax1.bar(['Non-Default', 'Default'], counts.values, color=['#2ecc71','#e74c3c'], width=0.5)
ax1.set_title('1. Default vs Non-Default Count', fontweight='bold')
ax1.set_ylabel('Number of Applicants')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
             f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontweight='bold')
ax1.set_ylim(0, counts.max()*1.15)

# --- Graph 2: Income vs Default ---
ax2 = axes[0, 1]
ax2.hist(df[df['default_risk']==0]['AMT_INCOME_K'], bins=50, alpha=0.65,
         color='#2ecc71', label='Non-Default', density=True)
ax2.hist(df[df['default_risk']==1]['AMT_INCOME_K'], bins=50, alpha=0.65,
         color='#e74c3c', label='Default', density=True)
ax2.set_title('2. Income Distribution vs Default', fontweight='bold')
ax2.set_xlabel('Annual Income (₹ thousands)')
ax2.set_ylabel('Density')
ax2.legend()
ax2.set_xlim(0, 5500)

# --- Graph 3: Loan Amount vs Default ---
ax3 = axes[1, 0]
data_0 = df[df['default_risk']==0]['AMT_CREDIT_K'].dropna()
data_1 = df[df['default_risk']==1]['AMT_CREDIT_K'].dropna()
bp = ax3.boxplot([data_0, data_1], tick_labels=['Non-Default','Default'],
                 patch_artist=True, widths=0.4)
bp['boxes'][0].set_facecolor('#2ecc71'); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#e74c3c'); bp['boxes'][1].set_alpha(0.7)
ax3.set_title('3. Loan Amount vs Default', fontweight='bold')
ax3.set_ylabel('Loan Amount (₹ thousands)')

# --- Graph 4: Correlation Heatmap ---
ax4 = axes[1, 1]
cols = ['default_risk','annual_income','loan_amount','emi_income_ratio',
        'cibil_score','missed_payments','bank_balance','years_employed']
labels = ['Default','Income','Loan Amt','EMI Ratio','CIBIL','Missed Pmts','Bank Bal','Yrs Emp']
corr = df[cols].corr()
corr.index = labels; corr.columns = labels
sns.heatmap(corr, ax=ax4, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, annot_kws={'size':8})
ax4.set_title('4. Correlation Heatmap', fontweight='bold')
ax4.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig('eda_graphs_indian.png', dpi=150, bbox_inches='tight')
plt.show()
print('All 4 graphs generated!')

## Step 4: Feature Understanding

| Feature | Role in CBES |
|---|---|
| **cibil_score** | Credit reliability — strongest predictor (corr: -0.334) |
| **missed_payments** | Repayment behaviour — 12 missed = 100% default rate |
| **emi_income_ratio** | Repayment capacity — >100% income is a red flag |
| **annual_income** | Earning power — weak alone, stronger in ratios |
| **loan_amount** | Risk exposure — how much debt is being taken on |
| **total_assets / bank_balance** | Safety net — absorbs income shocks |
| **years_employed** | Income stability — longer = more reliable |
| **employment_type** | Income predictability — salaried > self-employed > unemployed |
| **default_risk** | Target variable — 1 = defaulted, 0 = repaid |

## Step 5: CBES v2 Score

### Formula:
```
CBES = 0.35 × P1 (Credit Behaviour)
     + 0.25 × P2 (Repayment Capacity)
     + 0.20 × P3 (Financial Stability)
     + 0.12 × P4 (Employment Profile)
     + 0.08 × P5 (Loan Risk Exposure)
```
All components normalized to [0,1]. Weights sum to 1.00.

In [ ]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

# P1 — Credit Behaviour (35%)
cibil_norm   = normalize(df['cibil_score'])
missed_norm  = normalize(df['missed_payments'])
P1 = 0.6 * cibil_norm + 0.4 * (1 - missed_norm)

# P2 — Repayment Capacity (25%)
emi_clipped  = df['emi_income_ratio'].clip(upper=5.0)
P2 = 1 - normalize(emi_clipped)

# P3 — Financial Stability (20%)
P3 = 0.6 * normalize(df['total_assets']) + 0.4 * normalize(df['bank_balance'])

# P4 — Employment Profile (12%)
emp_map = {'Salaried': 1.0, 'Business Owner': 0.8, 'Self-Employed': 0.6, 'Unemployed': 0.2}
emp_score = df['employment_type'].map(emp_map)
P4 = 0.5 * emp_score + 0.5 * normalize(df['years_employed'])

# P5 — Loan Risk Exposure (8%)
P5 = 1 - normalize(df['loan_income_ratio'])

# Final CBES Score
df['CBES_SCORE'] = (
    0.35 * P1 +
    0.25 * P2 +
    0.20 * P3 +
    0.12 * P4 +
    0.08 * P5
).round(4)

# Decision logic
def cbes_decision(score):
    if score >= 0.65:   return 'Approve'
    elif score >= 0.35: return 'Defer'
    else:               return 'Reject'

df['CBES_DECISION'] = df['CBES_SCORE'].apply(cbes_decision)

# Show 10 sample applicants
sample_cols = ['applicant_id','annual_income','loan_amount','cibil_score',
               'missed_payments','employment_type','default_risk','CBES_SCORE','CBES_DECISION']
print('=== CBES v2 — Sample Applicants ===')
print(df[sample_cols].head(10).to_string(index=False))

print('\n=== Decision Distribution ===')
print(df['CBES_DECISION'].value_counts())

print('\n=== CBES Score Stats ===')
print(df['CBES_SCORE'].describe().round(3))

## Step 6: CBES vs Actual Default — Validation

In [ ]:
# How well does CBES decision align with actual default?
from sklearn.metrics import classification_report, confusion_matrix

# Map CBES decision to binary: Approve=0 (low risk), Reject=1 (high risk), Defer=0.5
df['CBES_BINARY'] = df['CBES_DECISION'].map({'Approve': 0, 'Defer': 0, 'Reject': 1})

print('CBES vs Actual Default - Accuracy check:')
print(f'Correct predictions: {(df["CBES_BINARY"] == df["default_risk"]).mean()*100:.1f}%')

# Default rate within each CBES band
print('\nDefault rate by CBES decision band:')
print(df.groupby('CBES_DECISION')['default_risk'].agg(['mean','count']).round(3))

# CBES score distribution by actual default
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df[df['default_risk']==0]['CBES_SCORE'], bins=40, alpha=0.65,
        color='#2ecc71', label='Non-Default', density=True)
ax.hist(df[df['default_risk']==1]['CBES_SCORE'], bins=40, alpha=0.65,
        color='#e74c3c', label='Default', density=True)
ax.axvline(0.35, color='orange', linestyle='--', label='Defer threshold (0.35)')
ax.axvline(0.65, color='green', linestyle='--', label='Approve threshold (0.65)')
ax.set_title('CBES Score Distribution by Actual Default', fontweight='bold')
ax.set_xlabel('CBES Score')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('cbes_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Validation chart saved!')

## Summary

- **4 EDA graphs** completed on Indian loan dataset (25,000 applicants)
- **CBES v2** applied with 5 data-driven pillars
- Decision bands: Approve (≥0.65) | Defer (0.35–0.64) | Reject (<0.35)
- CBES feeds both the ML model (Shweta) and the L2D router (Shreyash)